# Stratigraphic Ablation Study
Evaluates the contribution of each Stratigraphic component by disabling them one at a time.

**Requires A100 GPU** (~30-45 min total for 5 configs on Llama-3.2-1B-Instruct).

Produces:
1. Ablation table (PPL, delta PPL, compression ratio per config)
2. Per-layer zone assignment heatmap

In [1]:
import sys, os

# Match the pattern from 03-kv-bench-modal.ipynb:
# If running from a cloud notebook, clone the repo first.
# If running locally from within the repo, just add project root to path.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
# Check if we're already inside the PROJECT repo
if os.path.exists(os.path.join(project_root, 'kv_bench')):
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)
elif os.path.exists('PROJECT/kv_bench'):
    os.chdir('PROJECT')
    sys.path.insert(0, os.getcwd())
else:
    # Cloud environment: clone and install
    os.system('GIT_SSL_NO_VERIFY=1 git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git PROJECT')
    os.chdir('PROJECT')
    os.system('pip install -e .')
    sys.path.insert(0, os.getcwd())

print(f'Working dir: {os.getcwd()}')

import torch
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from kv_bench.runner import BenchmarkRunner
from kv_bench.device_config import auto_detect, DeviceConfig
from kv_bench.strategies import FullKVBaseline, StratigraphicStrategy
from streaming_attention.stratigraphic import StratigraphicConfig, ZONE_FP16, ZONE_INT8, ZONE_INT4, ZONE_EVICT

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

Cloning into 'PROJECT'...


Obtaining file:///root/PROJECT
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 74.7 MB/s eta 0:00:00


  DEPRECATION: Legacy editable install of streaming_attention==0.3.0 from file:///root/PROJECT (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457


  Running setup.py develop for streaming_attention



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Working dir: /root/PROJECT
CUDA available: False


## 1. Define ablation configurations

In [2]:
MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
MAX_SAMPLES = 100

# Each config tests removing one component
ablation_configs = {
    'Full Stratigraphic': StratigraphicConfig(),  # default: all components on
    'No anchors': StratigraphicConfig(anchor_budget=0.0, anchor_attn_percentile=1.0),
    'Uniform layer budget': StratigraphicConfig(lambda_=0.0),  # no inverse gradient
    'Quant-only (no eviction)': StratigraphicConfig(quant_only=True),
    'Equal zones': StratigraphicConfig(zone_surface=0.33, zone_shallow=0.33, zone_deep=0.34, lambda_=0.0),
}

print('Ablation configs defined:')
for name, cfg in ablation_configs.items():
    print(f'  {name}: surface={cfg.zone_surface}, shallow={cfg.zone_shallow}, '
          f'deep={cfg.zone_deep}, lambda={cfg.lambda_}, anchors={cfg.anchor_budget}')

Ablation configs defined:
  Full Stratigraphic: surface=0.2, shallow=0.4, deep=0.4, lambda=0.6, anchors=0.05
  No anchors: surface=0.2, shallow=0.4, deep=0.4, lambda=0.6, anchors=0.0
  Uniform layer budget: surface=0.2, shallow=0.4, deep=0.4, lambda=0.0, anchors=0.05
  Quant-only (no eviction): surface=0.2, shallow=0.4, deep=0.4, lambda=0.6, anchors=0.05
  Equal zones: surface=0.33, shallow=0.33, deep=0.34, lambda=0.0, anchors=0.05


In [3]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## 2. Run baseline (FullKV)

In [11]:
device_config = auto_detect()
print(f'Device config: {device_config}')

# Run baseline first
runner = BenchmarkRunner(
    model_name=MODEL,
    strategies=[FullKVBaseline()],
    device_config=device_config,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_samples=MAX_SAMPLES,
)
baseline_results = runner.run()
baseline_ppl = baseline_results[0].perplexity
print(f'\nBaseline PPL: {baseline_ppl:.2f}')

Device config: DeviceConfig(device='cpu', gpu_name='cpu', gpu_memory_gb=0, dtype=torch.float32, batch_size=1, max_seq_len=256, max_samples=10, stride=128, load_in_8bit=False, load_in_4bit=False)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

PPL (FullKV (baseline)):  95%|███████████████████████▊ | 40/42 [03:50<00:11,  5.76s/it, ppl=21.10, tokens=5248]


Baseline PPL: 21.02


## 3. Run each ablation config

In [12]:
ablation_results = {}

for name, cfg in ablation_configs.items():
    print(f'\n--- Running: {name} ---')
    strategy = StratigraphicStrategy(config=cfg)
    
    runner = BenchmarkRunner(
        model_name=MODEL,
        strategies=[strategy],
        device_config=device_config,
        dataset_name='wikitext',
        dataset_config='wikitext-2-raw-v1',
        split='test',
        max_samples=MAX_SAMPLES,
    )
    results = runner.run()
    r = results[0]
    ablation_results[name] = {
        'ppl': r.perplexity,
        'delta_ppl': r.perplexity - baseline_ppl,
        'compression': r.compression_ratio,
        'memory_mb': r.memory_kv_analytical_mb,
    }
    print(f'  PPL: {r.perplexity:.2f} (delta: {r.perplexity - baseline_ppl:+.2f}), '
          f'Compression: {r.compression_ratio:.2f}x')


--- Running: Full Stratigraphic ---


`torch_dtype` is deprecated! Use `dtype` instead!
PPL (Stratigraphic):  95%|███████████████████████████▌ | 40/42 [08:29<00:25, 12.74s/it, ppl=21.23, tokens=5248]


  PPL: 21.15 (delta: +0.14), Compression: 1.82x

--- Running: No anchors ---


PPL (Stratigraphic):  95%|███████████████████████████▌ | 40/42 [08:33<00:25, 12.84s/it, ppl=21.24, tokens=5248]


  PPL: 21.16 (delta: +0.14), Compression: 1.82x

--- Running: Uniform layer budget ---


...

  PPL: 21.11 (delta: +0.10), Compression: 1.82x

--- Running: Quant-only (no eviction) ---


PPL (Stratigraphic):  95%|███████████████████████████▌ | 40/42 [07:58<00:23, 11.95s/it, ppl=21.23, tokens=5248]


  PPL: 21.15 (delta: +0.14), Compression: 1.82x

--- Running: Equal zones ---


PPL (Stratigraphic):  95%|███████████████████████████▌ | 40/42 [07:24<00:22, 11.12s/it, ppl=21.17, tokens=5248]

  PPL: 21.08 (delta: +0.07), Compression: 1.73x


## 4. Print ablation table

In [13]:
print(f'\nBaseline (FullKV): PPL = {baseline_ppl:.2f}')
print()
print(f'{"Configuration":<30} {"PPL":>8} {"delta PPL":>10} {"Compression":>12}')
print('-' * 62)
for name, r in ablation_results.items():
    print(f'{name:<30} {r["ppl"]:>8.2f} {r["delta_ppl"]:>+10.2f} {r["compression"]:>11.2f}x')

# Save results
save_data = {'baseline_ppl': baseline_ppl, 'ablations': ablation_results}
save_path = Path('kv_bench/results/ablation_results.json')
save_path.parent.mkdir(parents=True, exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(save_data, f, indent=2)
print(f'\nSaved to {save_path.resolve()}')


Baseline (FullKV): PPL = 21.02

Configuration                       PPL  delta PPL  Compression
--------------------------------------------------------------
Full Stratigraphic                21.15      +0.14        1.82x
No anchors                        21.16      +0.14        1.82x
Uniform layer budget              21.11      +0.10        1.82x
Quant-only (no eviction)          21.15      +0.14        1.82x
Equal zones                       21.08      +0.07        1.73x

Saved to /root/PROJECT/kv_bench/results/ablation_results.json


## 5. Per-layer zone assignment heatmap
Run the full Stratigraphic strategy and capture zone assignments per layer and head.

In [14]:
# Run full stratigraphic once more to capture zone masks
strategy = StratigraphicStrategy(config=StratigraphicConfig())

runner = BenchmarkRunner(
    model_name=MODEL,
    strategies=[strategy],
    device_config=device_config,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_samples=10,  # small sample to get zone structure
)
_ = runner.run()

# Extract zone masks from the last evaluation window
seq_len = device_config.max_seq_len
device = next(runner._model.parameters()).device if hasattr(runner, '_model') else 'cuda'
zone_masks = strategy.get_zone_masks(seq_len, device)

if zone_masks is not None:
    print(f'Got zone masks for {len(zone_masks)} layers')
    # Compute dominant zone per (layer, head)
    num_layers = len(zone_masks)
    num_heads = zone_masks[0].shape[0]
    
    # For each (layer, head), compute fraction in each zone
    zone_fractions = np.zeros((num_layers, num_heads, 4))  # 4 zones
    for l in range(num_layers):
        zones = zone_masks[l].cpu().numpy()  # [H, seq_len]
        for h in range(num_heads):
            for z in range(4):
                zone_fractions[l, h, z] = (zones[h] == z).mean()
    
    print(f'Zone fractions shape: {zone_fractions.shape}')  # [layers, heads, 4]
else:
    print('No zone masks available (strategy may not have run with attention weights)')

PPL (Stratigraphic):  50%|████████████████                | 2/4 [00:30<00:30, 15.24s/it, ppl=16.63, tokens=384]

No zone masks available (strategy may not have run with attention weights)


In [15]:
if zone_masks is not None:
    # Plot 1: FP16 fraction heatmap (shows inverse layer budget)
    fig, axes = plt.subplots(1, 4, figsize=(16, 8), sharey=True)
    
    zone_names = ['FP16 (Surface)', 'INT8 (Shallow)', 'INT4 (Deep)', 'Evicted (Bedrock)']
    cmaps = ['Greens', 'YlOrBr', 'Oranges', 'Reds']
    
    for i, (ax, zname, cmap) in enumerate(zip(axes, zone_names, cmaps)):
        im = ax.imshow(zone_fractions[:, :, i], aspect='auto', cmap=cmap,
                       vmin=0, vmax=1.0, interpolation='nearest')
        ax.set_title(zname, fontsize=11)
        ax.set_xlabel('KV Head', fontsize=10)
        if i == 0:
            ax.set_ylabel('Layer', fontsize=10)
        ax.set_xticks(range(num_heads))
        plt.colorbar(im, ax=ax, shrink=0.6, label='Fraction')
    
    fig.suptitle(f'Stratigraphic Zone Assignment ({MODEL.split("/")[-1]})', fontsize=13, y=1.02)
    plt.tight_layout()
    
    save_path = Path('kv_bench/results/zone_heatmap.png')
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f'Saved to {save_path.resolve()}')
    plt.show()

In [16]:
if zone_masks is not None:
    # Plot 2: Stacked bar chart showing zone distribution per layer (averaged across heads)
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # Average across heads
    layer_zones = zone_fractions.mean(axis=1)  # [layers, 4]
    layers = np.arange(num_layers)
    
    colors = ['#2ca02c', '#ffbf00', '#ff7f0e', '#d62728']
    labels = ['FP16', 'INT8', 'INT4', 'Evicted']
    
    bottom = np.zeros(num_layers)
    for z in range(4):
        ax.bar(layers, layer_zones[:, z], bottom=bottom, color=colors[z],
               label=labels[z], width=0.8, edgecolor='white', linewidth=0.3)
        bottom += layer_zones[:, z]
    
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Fraction of Tokens', fontsize=12)
    ax.set_title(f'Stratigraphic Zone Distribution per Layer\n({MODEL.split("/")[-1]}, inverse layer budget)', fontsize=13)
    ax.legend(loc='upper left', fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(-0.5, num_layers - 0.5)
    
    plt.tight_layout()
    save_path = Path('kv_bench/results/zone_distribution_per_layer.png')
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f'Saved to {save_path.resolve()}')
    plt.show()

## 6. Summary
Print a LaTeX-ready ablation table.

In [17]:
print('\n% LaTeX table for report')
print(r'\begin{table}[h]')
print(r'\centering')
print(r'\caption{Stratigraphic ablation study on Llama-3.2-1B-Instruct (WikiText-2)}')
print(r'\label{tab:ablation}')
print(r'\begin{tabular}{lccc}')
print(r'\hline')
print(r'Configuration & PPL & $\Delta$PPL & Compression \\')
print(r'\hline')
print(f'FullKV (baseline) & {baseline_ppl:.2f} & --- & 1.0$\\times$ \\\\')
for name, r in ablation_results.items():
    print(f'{name} & {r["ppl"]:.2f} & {r["delta_ppl"]:+.2f} & {r["compression"]:.1f}$\\times$ \\\\')
print(r'\hline')
print(r'\end{tabular}')
print(r'\end{table}')


% LaTeX table for report
\begin{table}[h]
\centering
\caption{Stratigraphic ablation study on Llama-3.2-1B-Instruct (WikiText-2)}
\label{tab:ablation}
\begin{tabular}{lccc}
\hline
Configuration & PPL & $\Delta$PPL & Compression \\
\hline
FullKV (baseline) & 21.02 & --- & 1.0$\times$ \\
Full Stratigraphic & 21.15 & +0.14 & 1.8$\times$ \\
No anchors & 21.16 & +0.14 & 1.8$\times$ \\
Uniform layer budget & 21.11 & +0.10 & 1.8$\times$ \\
Quant-only (no eviction) & 21.15 & +0.14 & 1.8$\times$ \\
Equal zones & 21.08 & +0.07 & 1.7$\times$ \\
\hline
\end{tabular}
\end{table}
